In [1]:
# inlegalbert_bilstm_mha_crf_damix_full.py
#
# Architecture:
#   InLegalBERT  →  Sentence BiLSTM  →  MHA Pooling  →  Context BiLSTM
#                →  Linear  →  CRF
#                +  Auxiliary CE Loss
#                +  Discourse-Aware Mixup (DAMix) Loss   ← NEW
#
# DAMix integrations vs anti-overfitting baseline (v2):
#   D1. DiscourseAwareMixup module with separate aux_head (2-layer MLP).
#   D2. Role-compatibility matrix (pre-computed bool tensor, O(1) lookup).
#   D3. Position-proximity gate  |pos_i − pos_j| < DAMIX_POS_TAU (0.25).
#   D4. Rare-class pair oversampling  ×DAMIX_RARE_WEIGHT (3.0).
#   D5. Dominant-label convention: λ clamped to [0.5,1.0] → y_mix = y_i.
#   D6. Scheduled α annealing:  0.4 (epoch 1) → 0.1 (final epoch).
#   D7. DAMix operates on ctx_out (post-BiLSTM, 128-d) — richest repr.
#   D8. Combined loss: CRF + 0.2·CE + 0.3·DAMix
#   D9. DAMix disabled (alpha=0.0) during validation/test → zero cost.
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG  (unchanged from v2 except OUT_DIR)
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_damix_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 60
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4

WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD  = 0.05

# ═══════════════════════════════════════════════════════════
# DAMix HYPERPARAMETERS  ← NEW BLOCK
# ═══════════════════════════════════════════════════════════
DAMIX_WEIGHT      = 0.3   # coefficient: loss = CRF + 0.2·CE + 0.3·DAMix
#
# ALPHA SCHEDULE EXPLANATION:
#   Beta(α, α) with large α → distribution is wide → λ can be anywhere
#   in [0.5, 1.0] after clamping → STRONG MIXING (explore diverse pairs)
#
#   Beta(α, α) with small α → distribution piles mass near 0 and 1
#   → after clamping to [0.5,1.0], λ ≈ 1.0 nearly always
#   → WEAK MIXING (i.e. almost unchanged embeddings, subtle perturbation)
#
#   Strategy: start with strong mixing (wide search), anneal to weak mixing
#   (fine-tune) as training progresses.
DAMIX_ALPHA_START = 0.4   # strong mixes early (epoch 1)
DAMIX_ALPHA_END   = 0.1   # subtle mixes late  (final epoch)

# POSITION PROXIMITY GATE:
#   Normalised document position = sentence_index / (doc_length - 1)
#   Range: [0.0 (first sentence) … 1.0 (last sentence)]
#   Only mix pairs whose positions differ by less than this threshold.
#   τ = 0.25 means: two sentences can be at most 25% of the document apart.
#   WHY? FAC tends to be at the start, RATIO at the end — mixing them
#   across large positional gaps creates semantically incoherent embeddings.
DAMIX_POS_TAU     = 0.25

# Memory guard: after pair enumeration we subsample to at most this many.
DAMIX_MAX_PAIRS   = 64

# Rare-class pairs get this weight during weighted subsampling.
# Effectively raises the probability that a rare-class sentence participates
# in a mix, directly countering class imbalance.
DAMIX_RARE_WEIGHT = 3.0

# ROLE COMPATIBILITY MAP:
#   Keys = a role; values = the set of roles it may legally be mixed with.
#   Design principle: mix functionally similar roles (same argumentative
#   purpose in legal discourse). Mixing across incompatible roles would
#   create nonsense interpolations (e.g., PREAMBLE + RPC).
COMPATIBLE_ROLES = {
    "RATIO":          {"RATIO", "ANALYSIS"},
    "ANALYSIS":       {"ANALYSIS", "RATIO", "ARG_PETITIONER", "ARG_RESPONDENT"},
    "ARG_PETITIONER": {"ARG_PETITIONER", "ARG_RESPONDENT", "ANALYSIS"},
    "ARG_RESPONDENT": {"ARG_RESPONDENT", "ARG_PETITIONER", "ANALYSIS"},
    "FAC":            {"FAC", "PREAMBLE"},
    "PREAMBLE":       {"PREAMBLE", "FAC"},
    "PRE_RELIED":     {"PRE_RELIED", "PRE_NOT_RELIED"},
    "PRE_NOT_RELIED": {"PRE_NOT_RELIED", "PRE_RELIED"},
    "RLC":            {"RLC", "ISSUE"},
    "ISSUE":          {"ISSUE", "RLC"},
    "STA":            {"STA"},
    "RPC":            {"RPC"},
    "NONE":           {"NONE"},
}

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":          "InLegalBERT Encoder",
        "sent_bilstm":   "Sentence BiLSTM",
        "mha_pooling":   "Multi-Head Attn Pooling",
        "ctx_bilstm":    "Context BiLSTM",
        "classifier":    "Classifier Head",
        "crf":           "CRF",
        "damix_head":    "DAMix Module",      # ← NEW
        "dropout":       "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_trainable + total_frozen,
    })

    print("\n" + "=" * 72)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA + CRF + DAMix)")
    print("=" * 72)
    print(f"  {'Component':<32} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 72)
    for r in rows:
        if r["Component"] == "── TOTAL ──":
            print("─" * 72)
        print(f"  {r['Component']:<32} "
              f"{r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} "
              f"{r['Total Params']:>12,}")
    print("=" * 72)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []

        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))

        if not sents or len(sents) != len(labs):
            continue

        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# DISCOURSE-AWARE MIXUP MODULE  ← ENTIRELY NEW
# ═══════════════════════════════════════════════════════════
class DiscourseAwareMixup(nn.Module):
    """
    Generates synthetic sentence embeddings by interpolating
    embeddings of discourse-compatible sentence pairs, then
    trains a lightweight auxiliary head on those synthetic samples.

    Three compatibility gates (ALL must pass):
      Gate 1 — Role compatibility  : labels must be in COMPATIBLE_ROLES
      Gate 2 — Position proximity  : |pos_i - pos_j| < DAMIX_POS_TAU
      Gate 3 — Valid labels only   : neither index has label == -100

    Loss is a standard CrossEntropyLoss on the synthetic embeddings
    via a dedicated 2-layer MLP (aux_head), so DAMix gradients do NOT
    corrupt the CRF emission head.
    """

    def __init__(self, embed_dim: int, num_labels: int, rare_ids=None):
        super().__init__()
        self.embed_dim  = embed_dim
        self.num_labels = num_labels
        self.rare_ids   = set(rare_ids) if rare_ids else set()

        # ── Separate head (keeps DAMix noisy gradients away from CRF) ──
        # WHY SEPARATE? During early training, DAMix mixes random-ish
        # embeddings and produces noisy gradients. If those gradients
        # flowed into the shared emission Linear, they would destabilise
        # CRF learning. The separate head absorbs all DAMix gradients.
        self.aux_head = nn.Sequential(
            nn.LayerNorm(embed_dim),             # stabilise mixed embeddings
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim // 2, num_labels),
        )

        # Auxiliary CE loss (label smoothing not used here because we
        # already enforce dominant-label convention; smoothing would
        # add unnecessary noise on top of the mixing noise).
        self.ce = nn.CrossEntropyLoss(ignore_index=-100)

        # ── Pre-compute role-compatibility bool matrix (NUM_LABELS × NUM_LABELS)
        # Instead of doing string lookups inside the hot loop, we build a
        # boolean matrix once and look up compat_matrix[li, lj] in O(1).
        compat = torch.zeros(num_labels, num_labels, dtype=torch.bool)
        for role, compat_set in COMPATIBLE_ROLES.items():
            i = label2id.get(role, -1)
            if i < 0:
                continue
            for other in compat_set:
                j = label2id.get(other, -1)
                if j >= 0:
                    compat[i, j] = True
                    compat[j, i] = True    # symmetric: if A can mix with B, B can mix with A
        self.register_buffer("compat_matrix", compat)
        # register_buffer: moves tensor with .to(device) but is NOT a parameter.

    # ── Lambda sampler ───────────────────────────────────────
    @staticmethod
    def _sample_lambda(alpha: float, n: int, device) -> torch.Tensor:
        """
        Sample λ ~ Beta(α, α), then clamp to [0.5, 1.0].

        WHY CLAMP?
          Standard Mixup uses λ anywhere in [0,1].
          Here we enforce λ ≥ 0.5 so that sentence i is ALWAYS
          the dominant contributor. This lets us safely use y_i as
          the hard label — CRF requires integer labels, not soft ones.

          If λ < 0.5 for a sampled pair (i, j), we flip:
            max(λ, 1-λ) ≥ 0.5  ← this is what torch.max does below.
          After the flip, the "primary" sentence (with weight ≥ 0.5)
          is always i. Label = y_i.

        WHY Beta(α, α)?
          Symmetric Beta ensures neither sentence is systematically
          favoured. After clamping the effective range is [0.5, 1.0].
          α=0.4 → most λ in [0.55, 0.95] (meaningful mixing).
          α=0.1 → most λ near 1.0 (nearly no mixing; subtle perturbation).
        """
        if alpha <= 0.0:
            return torch.ones(n, device=device)
        dist = torch.distributions.Beta(
            torch.tensor(alpha, device=device),
            torch.tensor(alpha, device=device),
        )
        lam = dist.sample((n,))
        return torch.max(lam, 1.0 - lam)    # enforce dominant-label convention

    # ── Pair builder ─────────────────────────────────────────
    def _build_valid_pairs(
        self,
        labels: torch.Tensor,      # (N_valid,) int tensor, all ≥ 0
        positions: torch.Tensor,   # (N_valid,) float tensor in [0, 1]
    ):
        """
        Enumerate all O(N²) candidate pairs and filter by the three gates.
        Returns (pairs, weights) where pairs = list of (i, j) tuples.

        N is capped at 128 to keep runtime tractable.
        At 128 sentences the worst case is 128*127/2 = 8128 pairs, each
        checked in microseconds (pure Python with numpy arrays).
        """
        N_cap  = min(labels.shape[0], 128)
        lab_np = labels[:N_cap].cpu().numpy().astype(int)
        pos_np = positions[:N_cap].cpu().numpy().astype(float)

        pairs, weights = [], []

        for i in range(N_cap):
            li = lab_np[i]
            pi = pos_np[i]
            for j in range(i + 1, N_cap):
                lj = lab_np[j]
                pj = pos_np[j]

                # Gate 1: position proximity
                if abs(pi - pj) >= DAMIX_POS_TAU:
                    continue

                # Gate 2: role compatibility (O(1) matrix lookup)
                if not self.compat_matrix[li, lj].item():
                    continue

                # Passed both gates → valid pair
                # Weight: rare-class pairs are oversampled
                w = DAMIX_RARE_WEIGHT if (li in self.rare_ids or
                                          lj in self.rare_ids) else 1.0
                pairs.append((i, j))
                weights.append(w)

        return pairs, weights

    # ── Main forward ─────────────────────────────────────────
    def forward(
        self,
        embeddings: torch.Tensor,   # (N_valid, embed_dim) — post-BiLSTM ctx vectors
        labels:     torch.Tensor,   # (N_valid,) int — ground truth class indices
        positions:  torch.Tensor,   # (N_valid,) float in [0,1] — normalised position
        alpha:      float = 0.2,    # Beta distribution shape (scheduled externally)
    ) -> torch.Tensor:
        """
        Full DAMix forward pass.
        Returns a scalar loss (0.0 if no valid pairs found).

        Flow:
          1. Build valid pairs (role + position gates).
          2. Subsample to DAMIX_MAX_PAIRS via weighted sampling.
          3. Sample λ ~ Beta(α,α) clamped to [0.5, 1.0].
          4. Interpolate:  ẽ = λ·E_i + (1−λ)·E_j
          5. Dominant label:  ỹ = y_i  (because λ ≥ 0.5 → i is dominant)
          6. Forward through aux_head → CE loss.
        """
        device = embeddings.device

        # ── Step 1: Build valid pairs ─────────────────────────
        pairs, weights = self._build_valid_pairs(labels, positions)

        if not pairs:
            # No valid pairs in this batch → return zero loss (no gradients).
            # This can happen when the batch has very few sentences or the
            # document is a very short document with only one role.
            return torch.tensor(0.0, device=device, requires_grad=True)

        # ── Step 2: Subsample ─────────────────────────────────
        if len(pairs) > DAMIX_MAX_PAIRS:
            w_arr = np.array(weights, dtype=np.float32)
            w_arr /= w_arr.sum()                          # normalise to probabilities
            chosen_idx = np.random.choice(
                len(pairs), DAMIX_MAX_PAIRS, replace=False, p=w_arr
            )
            pairs = [pairs[k] for k in chosen_idx]
            # Note: weights are only needed for subsampling; drop them after.

        idx_i = torch.tensor([p[0] for p in pairs], dtype=torch.long, device=device)
        idx_j = torch.tensor([p[1] for p in pairs], dtype=torch.long, device=device)

        # ── Step 3: Sample λ ──────────────────────────────────
        # Shape: (n_pairs, 1) for broadcasting over embed_dim dimension
        lam = self._sample_lambda(alpha, len(pairs), device).unsqueeze(-1)

        # ── Step 4: Mix embeddings ────────────────────────────
        # ẽ = λ·E_i + (1−λ)·E_j
        # Both E_i and E_j are ctx_out vectors: (n_pairs, 128)
        e_i   = embeddings[idx_i]   # primary (dominant) embedding
        e_j   = embeddings[idx_j]   # secondary embedding
        e_mix = lam * e_i + (1.0 - lam) * e_j   # (n_pairs, embed_dim)

        # ── Step 5: Dominant label ────────────────────────────
        # Because λ ≥ 0.5, sentence i contributes more than half the signal.
        # We assign y_mix = y_i (hard integer label, CRF compatible).
        # This avoids the soft-label problem where the fractional target
        # cannot be fed to standard CE or CRF.
        y_mix = labels[idx_i]   # (n_pairs,) int64

        # ── Step 6: Auxiliary CE via separate head ────────────
        logits = self.aux_head(e_mix)         # (n_pairs, num_labels)
        return self.ce(logits, y_mix)         # scalar


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    """
    Pipeline:
      [per sentence]
        Token IDs → InLegalBERT → token embeddings (B*T, L, 768)
                  → Sentence BiLSTM → contextual token vectors (B*T, L, 256)
                  → MHA Pooling → sentence vector (B*T, 256)
      [per document]
        sentence vectors (B, T, 256)
                  → Context BiLSTM → ctx_out (B, T, 128)
                  → Linear head → emissions (B, T, NUM_LABELS)
                  → CRF → predicted sequence

    Losses:
      CRF loss (sequence-level)
      + AUX_CE_WEIGHT × CE loss (dense token-level, label smoothed)
      + DAMIX_WEIGHT  × DAMix loss (synthetic rare-class embeddings)
    """

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        rare_ids         = None,         # ← passed in for DAMix
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2     # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        # Store ctx_out_dim as attribute: forward() needs it for DAMix reshape
        self.ctx_out_dim = ctx_lstm_hidden * 2     # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        # Auxiliary CE loss with label smoothing
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

        # ── DAMix module ──────────────────────────────────────
        # WHY embed_dim = ctx_out_dim (128)?
        #   ctx_out is the richest representation available:
        #   it has absorbed BERT token meaning, sentence-level BiLSTM
        #   context, MHA pooling, and now document-level context via
        #   the second BiLSTM. Mixing here is semantically the most
        #   meaningful interpolation point in the pipeline.
        self.damix_head = DiscourseAwareMixup(
            embed_dim  = self.ctx_out_dim,
            num_labels = num_labels,
            rare_ids   = rare_ids,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False

        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False

        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1} "
              f"({n_freeze} layers).")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0

        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
        damix_alpha:    float = 0.0,        # ← NEW: 0.0 = DAMix disabled
    ):
        """
        damix_alpha: float
            Scheduled Beta(α,α) shape parameter for DAMix.
            Pass 0.0 at eval/test time to disable DAMix entirely.
            Trainer._get_damix_alpha() computes this per epoch.
        """
        # ── Sentence encoding ─────────────────────────────────
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        # ── Context enrichment BiLSTM ─────────────────────────
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)    # (B, T, 128)
        emissions = self.classifier(ctx_out) # (B, T, NUM_LABELS)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        # ── Build validity mask ───────────────────────────────
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        # ── Training: compute combined loss ───────────────────
        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            # 1) CRF loss (sequence-level structured loss)
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            # 2) Auxiliary dense CE loss (per-token, label-smoothed)
            #    Provides dense gradient signal to speed up convergence.
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            # 3) DAMix loss ─────────────────────────────────────
            #    Only run if alpha > 0 (training) and there is a valid ctx_out.
            if damix_alpha > 0.0:
                # Flatten ctx_out and labels to 1-D (one row per sentence)
                # then keep only the non-padded positions (mask==True).
                flat_mask_1d = mask.reshape(-1)             # (B*T,)
                flat_ctx     = ctx_out.reshape(-1, self.ctx_out_dim)   # (B*T, 128)
                flat_labels  = labels.reshape(-1)           # (B*T,)

                valid_emb = flat_ctx[flat_mask_1d]          # (N_valid, 128)
                valid_lbl = flat_labels[flat_mask_1d]       # (N_valid,)

                # Compute normalised position for every valid sentence.
                # pos = t / (doc_length - 1).   doc_length = lengths[b_i].
                # This tells DAMix where in the document the sentence sits.
                pos_list = []
                for b_i in range(B2):
                    T_doc = int(lengths[b_i].item()) if lengths is not None else T2
                    for t_i in range(T2):
                        if mask[b_i, t_i]:
                            pos_list.append(float(t_i) / max(T_doc - 1, 1))

                valid_pos = torch.tensor(
                    pos_list, dtype=torch.float32, device=ctx_out.device
                )  # (N_valid,)

                damix_loss = self.damix_head(
                    embeddings = valid_emb,
                    labels     = valid_lbl,
                    positions  = valid_pos,
                    alpha      = damix_alpha,
                )
            else:
                damix_loss = torch.tensor(0.0, device=ctx_out.device)

            # ── Combined loss ─────────────────────────────────
            loss = (
                crf_loss
                + AUX_CE_WEIGHT * ce_loss
                + DAMIX_WEIGHT  * damix_loss
            )
            return loss, emissions

        # ── Inference: CRF decode (no DAMix) ─────────────────
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        str_trues, str_preds,
        labels=LABELS,
        digits=4,
        zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1":            macro_f1,
        "micro_f1":            micro_f1,
        "weighted_f1":         weighted_f1,
        "macro_precision":     macro_prec,
        "micro_precision":     micro_prec,
        "weighted_precision":  weighted_prec,
        "macro_recall":        macro_rec,
        "micro_recall":        micro_rec,
        "weighted_recall":     weighted_rec,
        "rare_f1":             rare_f1,
        "rare_precision":      rare_prec,
        "rare_recall":         rare_rec,
        "per_class_metrics":   per_class_metrics,
        "accuracy":            acc,
        "cls_report":          cls_report,
        "cm":                  cm,
        "all_preds":           all_preds,
        "all_trues":           all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    # ── Alpha scheduler ───────────────────────────────────────
    @staticmethod
    def _get_damix_alpha(epoch: int, num_epochs: int) -> float:
        """
        Linear annealing from DAMIX_ALPHA_START → DAMIX_ALPHA_END.

        WHY LINEAR ANNEAL?
          Early training: model representations are poorly calibrated.
          Wide Beta(0.4, 0.4) → diverse λ values → strong mixes →
          the model sees many synthetic embeddings and is forced to
          learn a representation where compatible roles are nearby.

          Late training: representations are refined.
          Narrow Beta(0.1, 0.1) → λ ≈ 1 most of the time →
          very subtle perturbations → acts like a weak form of
          Gaussian noise / dropout regularisation, not structural mixing.

        Returns 0.0 if num_epochs == 1 (edge case guard).
        """
        if num_epochs <= 1:
            return DAMIX_ALPHA_START
        progress = (epoch - 1) / (num_epochs - 1)          # 0.0 … 1.0
        alpha = DAMIX_ALPHA_START + progress * (DAMIX_ALPHA_END - DAMIX_ALPHA_START)
        return float(alpha)

    def build_optimizer(self):
        """
        Layer-wise LR decay for BERT encoder:
          Pooler              → BERT_LR
          Layer 11 (top)      → BERT_LR × 0.9^0  = BERT_LR
          Layer 10            → BERT_LR × 0.9^1
          ...
          Layer 8 (first fine-tuned) → BERT_LR × 0.9^3
        Head modules          → HEAD_LR
        """
        param_groups = []

        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
            self.model.damix_head,    # ← DAMix head gets HEAD_LR too
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))

        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                    damix_alpha=0.0,   # ← no DAMix during val loss computation
                )
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None

        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            # ── Compute scheduled alpha for this epoch ────────
            damix_alpha = self._get_damix_alpha(epoch, num_epochs)

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                    damix_alpha=damix_alpha,    # ← pass scheduled alpha
                )

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            # Flush remaining gradients if batch count is odd
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan_steps={nan_steps}]" if nan_steps > 0 else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"val_acc: {val_metrics['accuracy']:.4f} | "
                f"α_damix: {damix_alpha:.3f} | "
                f"time: {epoch_train_time:.1f}s"
                f" | ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{nan_info}"
            )

            row = {
                "epoch":                   epoch,
                "damix_alpha":             damix_alpha,    # ← logged for analysis
                "train_loss":              avg_train_loss,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_train_time,
                "nan_steps":               nan_steps,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping triggered after epoch {epoch} "
                      f"(no improvement for {early_stopper.patience} epochs).\n")
                break

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time : {total_train_time/60:.2f} min "
              f"({total_train_time:.1f} s)  —  {actual_epochs} epochs run")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "num_epochs_max":          num_epochs,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)

        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)

                # damix_alpha=0.0 → DAMix is completely skipped at inference
                decoded, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths,
                    damix_alpha=0.0,
                )
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)

            print(f"\n⏱  Inference ({split_name}): "
                  f"{total_infer_time:.2f}s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f}ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
            "freeze_layers":    BERT_FREEZE_LAYERS,
            "damix_weight":     DAMIX_WEIGHT,
            "damix_alpha_start": DAMIX_ALPHA_START,
            "damix_alpha_end":   DAMIX_ALPHA_END,
            "damix_pos_tau":     DAMIX_POS_TAU,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        ax.set_title("Training vs Validation Loss (CRF + CE + DAMix)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1 Scores"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close()

        # ── DAMix alpha schedule plot ─────────────────────────
        if "damix_alpha" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(epochs, hist_df["damix_alpha"], color="purple", marker="o")
            ax.set_title("DAMix Alpha Schedule (annealing)")
            ax.set_xlabel("Epoch"); ax.set_ylabel("α (Beta shape)")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "damix_alpha_schedule.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close()

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(
                hist_df["epoch_train_time_s"].mean(), color="red", linestyle="--",
                label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s"
            )
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close()

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close()

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close()


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 70)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + DAMix)")
    print("=" * 70)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 70)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 70)
    for label, key in rows:
        sep = "─" * 70 if key == "rare_f1" else ""
        if sep:
            print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 70)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-4) → Sent-BiLSTM(128) → "
          "MHA(4) → Ctx-BiLSTM(64) → Linear → CRF + CE + DAMix")
    print(f"\nDAMix settings:")
    print(f"  Weight         : {DAMIX_WEIGHT}")
    print(f"  Alpha start    : {DAMIX_ALPHA_START}")
    print(f"  Alpha end      : {DAMIX_ALPHA_END}")
    print(f"  Position τ     : {DAMIX_POS_TAU}")
    print(f"  Max pairs      : {DAMIX_MAX_PAIRS}")
    print(f"  Rare boost     : {DAMIX_RARE_WEIGHT}×\n")

    print("Loading JSONL files...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        rare_ids         = rare_ids,    # ← pass rare ids into model for DAMix
    )

    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)

    print(f"\nStarting training (max {NUM_EPOCHS} epochs, "
          f"early stop patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + DAMix\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])
    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + DAMix\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])
    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        rows_pc = []
        for lbl in LABELS:
            pc = mets["per_class_metrics"][lbl]
            rows_pc.append({
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        pc["f1"],
                "precision": pc["precision"],
                "recall":    pc["recall"],
            })
        pd.DataFrame(rows_pc).to_csv(
            os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False
        )

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA + CRF + DAMix",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
            "total_params":     total_trainable + total_frozen,
        },
        "damix": {
            "weight":      DAMIX_WEIGHT,
            "alpha_start": DAMIX_ALPHA_START,
            "alpha_end":   DAMIX_ALPHA_END,
            "pos_tau":     DAMIX_POS_TAU,
            "max_pairs":   DAMIX_MAX_PAIRS,
            "rare_weight": DAMIX_RARE_WEIGHT,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-4) → Sent-BiLSTM(128) → MHA(4) → Ctx-BiLSTM(64) → Linear → CRF + CE + DAMix

DAMix settings:
  Weight         : 0.3
  Alpha start    : 0.4
  Alpha end      : 0.1
  Position τ     : 0.25
  Max pairs      : 64
  Rare boost     : 3.0×

Loading JSONL files...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7 (8 layers).
🔥 BERT layers trainable: layers 8-11 (4 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA + CRF + DAMix)
  Component                             Trainable     Frozen        Total
------------------------------------------------------------------------
  InLegalBERT Encoder                  28,942,080 80,540,160  109,482,240
  Sentence BiLSTM                       1,314,816          0    1,314,816
  Multi-Head Attn Pooling                 197,120          0      197,120
  Context BiLSTM                          264,192          0      264,192
  Classifier Head                           9,101          0        9,101
  CRF                                         195          0          195
  DAMix Module                              9,357          0        9,357
  Dropout                                       0          0            0
────────────────────────────────────────────────────────────────────────